# Notebook 03 — Baselines Clássicos

Este notebook é a camada de **apresentação e análise** do pipeline clássico.
Toda a lógica de treino vive em `src/` e é orquestrada pelo DVC.

**O que acontece aqui:**
1. Inspecionar o `ColumnTransformer` (quais features, quais transformadores)
2. Rodar o pipeline completo via script e observar os logs
3. Carregar os artefatos gerados e analisar os resultados
4. Comparar LogReg × LinearSVC × RandomForest nas métricas de negócio

**KPI do negócio:** recall@crítico ≥ 0.75 (não deixar passar incidentes críticos)

---

### Por que scripts em vez de código no notebook?

| Notebook | Script |
|----------|--------|
| Resultado depende de execução manual | Reproduzível via `dvc repro` |
| Sem rastreamento de parâmetros | MLflow loga tudo automaticamente |
| Difícil de versionar e testar | Módulo importável, testável |
| `Kernel > Restart & Run All` é o deploy | `dvc repro` é o deploy |

## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import json
import joblib
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

from src.features.build import build_feature_pipeline, prepare_dataframe
from src.models.classic import MODEL_BUILDERS, get_param_grid
from src.evaluation.metrics import (
    classification_metrics,
    per_class_report,
    compare_models_table,
    plot_confusion_matrix,
    plot_metrics_comparison,
)

with open(ROOT / 'params.yaml') as f:
    params = yaml.safe_load(f)

with open(ROOT / 'configs/eda.yaml') as f:
    eda_cfg = yaml.safe_load(f)

print('ROOT:', ROOT)
print('Modelos:', list(MODEL_BUILDERS.keys()))

## 1. Inspecionar o pipeline de features

Antes de qualquer treino, é importante entender **o que entra no modelo**.
O `ColumnTransformer` é construído 100% a partir do `params.yaml`.

In [ ]:
feature_pipeline = build_feature_pipeline(params)
print(feature_pipeline)

In [ ]:
# O que está configurado?
print('=== Features de texto (TF-IDF) ===')
tfidf_cfg = params['features']['tfidf']
for k, v in tfidf_cfg.items():
    print(f'  {k}: {v}')

print('\n=== Features categóricas (OHE) ===')
for feat in params['features']['categorical']:
    print(f'  {feat}')

print('\n=== Features numéricas ===')
for feat in params['features']['numeric']:
    print(f'  {feat}')

## 2. Rodar o pipeline via DVC

Em produção (ou no terminal), o comando é:

```bash
dvc repro train_classic
```

Aqui no notebook, rodamos diretamente para visualizar os logs:

In [ ]:
# Roda o script diretamente (equivalente a `dvc repro train_classic`)
# Descomente quando quiser re-treinar do zero

# import subprocess
# result = subprocess.run(
#     ['python', str(ROOT / 'src/pipeline/train_classic.py')],
#     capture_output=False, text=True
# )

print('Para rodar o pipeline:')
print('  dvc repro train_classic')
print('  # ou:')
print('  python src/pipeline/train_classic.py')

## 3. Carregar resultados

In [ ]:
metrics_path = ROOT / 'reports/metrics_classic.json'

if not metrics_path.exists():
    print('⚠️  Execute o pipeline primeiro: dvc repro train_classic')
else:
    with open(metrics_path) as f:
        raw_metrics = json.load(f)

    # Reconstruir por modelo
    model_names = list(MODEL_BUILDERS.keys())
    results = {}
    for name in model_names:
        results[name] = {
            k.replace(f'{name}__', ''): v
            for k, v in raw_metrics.items()
            if k.startswith(f'{name}__')
        }

    print(f"Melhor modelo: {raw_metrics.get('best_model')}")
    print(f"KPI (recall@crítico): {results[raw_metrics['best_model']]['recall_critico']:.4f}")

## 4. Comparação de modelos

In [ ]:
if 'results' in dir():
    table = compare_models_table(results)
    print(table.to_string())

In [ ]:
if 'results' in dir():
    fig = plot_metrics_comparison(results)
    plt.show()

## 5. Análise do melhor modelo

In [ ]:
best_model_path = ROOT / 'models/classic/best_model.joblib'

if not best_model_path.exists():
    print('⚠️  Execute o pipeline primeiro.')
else:
    best_model = joblib.load(best_model_path)
    print('Melhor modelo carregado:', type(best_model.named_steps['classifier']).__name__)
    print('Parâmetros:')
    print(best_model.named_steps['classifier'].get_params())

In [ ]:
# Confusion matrix no test set
if 'best_model' in dir():
    test = pd.read_parquet(ROOT / eda_cfg['paths']['test'])
    X_test = prepare_dataframe(test, params)
    y_test = test[params['base']['target_col']]

    y_pred = best_model.predict(X_test)

    best_name = raw_metrics.get('best_model', 'best')
    fig = plot_confusion_matrix(
        y_test, y_pred,
        title=f'Confusion Matrix — {best_name}',
        class_order=params['base']['class_order']
    )
    plt.show()

In [ ]:
# Report por classe
if 'y_pred' in dir():
    report = per_class_report(y_test, y_pred, params['base']['class_order'])
    display(report.style.highlight_max(subset=['recall', 'f1'], color='#c8e6c9')
                        .highlight_min(subset=['recall', 'f1'], color='#ffcdd2'))

## 6. Feature importance (LogReg / LinearSVC)

Para modelos lineares, os coeficientes revelam **quais tokens e features empurram cada classe**.

In [ ]:
def plot_top_features(pipeline, class_order, top_n=15):
    import numpy as np
    clf = pipeline.named_steps['classifier']
    preprocessor = pipeline.named_steps['preprocessor']

    if not hasattr(clf, 'coef_'):
        print('Modelo não tem coef_ (RandomForest?) — use feature_importances_')
        return

    feature_names = preprocessor.get_feature_names_out()
    coef = clf.coef_  # shape: (n_classes, n_features) ou (n_features,)

    if coef.ndim == 1:
        coef = coef.reshape(1, -1)

    n_classes = len(class_order) if coef.shape[0] > 1 else 1
    fig, axes = plt.subplots(1, n_classes, figsize=(5 * n_classes, 5))
    if n_classes == 1:
        axes = [axes]

    for i, (ax, cls) in enumerate(zip(axes, class_order[:n_classes])):
        idx = coef[i].argsort()[::-1][:top_n]
        ax.barh(feature_names[idx][::-1], coef[i][idx][::-1], color='#1976D2', edgecolor='white')
        ax.set_title(f'Classe: {cls}', fontsize=10)
        ax.set_xlabel('Coeficiente')
        ax.tick_params(axis='y', labelsize=7)

    fig.suptitle('Top features por classe (coeficientes do modelo linear)', fontweight='bold')
    fig.tight_layout()
    plt.show()

if 'best_model' in dir():
    plot_top_features(best_model, params['base']['class_order'])

## 7. MLflow UI

Para visualizar todos os experimentos:

```bash
mlflow ui --backend-store-uri mlruns
# Acesse: http://localhost:5000
```

Você vai ver:
- Todos os runs com seus hiperparâmetros
- Métricas comparadas lado a lado
- Artefatos (modelos `.joblib`, confusion matrices)
- Qual run produziu o `best_model.joblib`

## 8. Explainable AI com SHAP

**Por que isso importa?**  
Coeficientes mostram importância *global* — o que o modelo aprendeu em geral.  
SHAP vai além: mostra **por que o modelo classificou *este* relato específico como crítico**.

> "O operador relatou chama aberta próxima ao manifold de gás." → SHAP aponta exatamente `chama`, `aberta`, `gás` como os tokens que empurraram para *crítico*.

Isso transforma um modelo em ferramenta utilizável por analistas de segurança.

### Como o SHAP funciona com TF-IDF + LogReg?

O `LinearExplainer` do SHAP aproveita a linearidade do modelo:

```
output = Σ (coef_i × tfidf_i)
```

Para cada predição, SHAP decompõe essa soma em contribuições por token — exatamente o que aparece no waterfall plot.

## 9. Próximos passos

| Tier | Notebook | O que muda |
|------|----------|------------|
| Classic + SHAP (este) | NB03 | TF-IDF + LogReg/SVM/RF + explicabilidade por instância |
| spaCy (BOW → tok2vec) | NB04 | Redes neurais profundas em português + hibridização |
| LLM | NB05 | Zero-shot e few-shot com Claude — sem treino |
| Benchmark | NB06 | Mesmo test set, mesmas métricas — comparação justa + Gradio demo |

**Pergunta para o próximo notebook:** o spaCy com tok2vec bate o recall@crítico do melhor baseline clássico?  
E quanto a arquitetura mais profunda muda o comportamento do modelo híbrido?

### 8.4 Bar chart global — Top tokens críticos vs. não-críticos

Quais tokens têm **maior impacto absoluto médio** na classificação de *crítico*?  
Esse gráfico é o que você mostraria para um analista de segurança como "o que o modelo considera arriscado".

In [ ]:
if 'shap_values' in dir() and 'shap_critico' in dir():
    # Falsos negativos: verdadeiro=crítico mas modelo não detectou
    fn_mask = (np.array(y_test) == 'critico') & (np.array(y_pred) != 'critico')
    fn_indices = np.where(fn_mask)[0]

    print(f'Total de falsos negativos (crítico não detectado): {fn_indices.sum()} de {(np.array(y_test)=="critico").sum()} críticos')

    if len(fn_indices) > 0:
        n_show = min(2, len(fn_indices))
        print(f'\nAnalisando {n_show} falso(s) negativo(s):\n')

        for rank, idx in enumerate(fn_indices[:n_show]):
            texto_original = test.iloc[idx][params['features']['text_col']]
            pred_label = y_pred[idx]
            print(f'--- Falso Negativo #{rank+1} (true=crítico, pred={pred_label}) ---')
            print(f'{texto_original[:200]}...' if len(texto_original) > 200 else texto_original)
            print()

            shap.plots.waterfall(shap_critico[idx], max_display=12, show=False)
            plt.title(f'SHAP — Falso Negativo #{rank+1} (pred={pred_label}, deveria ser crítico)', pad=12)
            plt.tight_layout()
            plt.show()
    else:
        print('Nenhum falso negativo — modelo capturou todos os críticos!')

### 8.3 Análise de falsos negativos — O que o modelo *não* detectou?

Falsos negativos (true=crítico, pred≠crítico) são o erro mais caro neste domínio (até R$ 3.2M/caso).  
SHAP nos diz **por que o modelo não detectou** — quais features estavam ausentes ou com peso insuficiente.

In [ ]:
if 'shap_values' in dir() and 'shap_critico' in dir():
    # Seleciona exemplos preditos como crítico pelo modelo
    critico_mask = np.array(y_pred) == 'critico'
    critico_indices = np.where(critico_mask)[0]

    if len(critico_indices) == 0:
        print('Nenhum relato classificado como crítico no test set.')
    else:
        # Mostra os 3 primeiros casos críticos detectados
        n_show = min(3, len(critico_indices))
        print(f'Mostrando waterfall para {n_show} relatos classificados como crítico:\n')

        for rank, idx in enumerate(critico_indices[:n_show]):
            texto_original = test.iloc[idx][params['features']['text_col']]
            true_label = y_test.iloc[idx]
            print(f'--- Relato #{rank+1} (true={true_label}) ---')
            print(f'{texto_original[:200]}...' if len(texto_original) > 200 else texto_original)
            print()

            shap.plots.waterfall(shap_critico[idx], max_display=12, show=False)
            plt.title(f'SHAP Waterfall — Relato #{rank+1} (true={true_label}, pred=crítico)', pad=12)
            plt.tight_layout()
            plt.show()

### 8.2 Explicação individual — Waterfall plot

O waterfall mostra **por que um relato específico recebeu aquela classificação**.  
Cada barra é um token ou feature, empurrando para cima (a favor de *crítico*) ou para baixo (contra).

Vamos olhar os relatos que o modelo **classificou como crítico** — os que mais impactam segurança.

In [ ]:
if 'shap_values' in dir() and hasattr(clf, 'classes_'):
    classes_model = list(clf.classes_)
    
    # Para modelos multi-classe, shap_values tem shape (n, features, n_classes)
    # Para binário ou OvR, pode ser (n, features) — detectamos dinamicamente
    is_multiclass = shap_values.values.ndim == 3

    if is_multiclass:
        # Foco no índice da classe 'critico'
        critico_idx = classes_model.index('critico') if 'critico' in classes_model else -1
        shap_critico = shap_values[:, :, critico_idx]
        title_suffix = "classe: crítico"
    else:
        shap_critico = shap_values
        title_suffix = "modelo (OvR)"

    # Beeswarm: top 20 features mais impactantes para 'crítico'
    shap.plots.beeswarm(
        shap_critico,
        max_display=20,
        show=False,
    )
    plt.title(f'SHAP — importância global ({title_suffix})', pad=14)
    plt.tight_layout()
    plt.show()

### 8.1 Importância global — Beeswarm plot

O beeswarm mostra a distribuição dos valores SHAP **de todos os exemplos do test set** para cada feature.  
Cada ponto = um relato. Cor quente = feature com valor alto (ex.: palavra presente). Posição no eixo X = impacto na predição.

> **Leitura:** tokens à direita empurram para a classe; à esquerda afastam.

In [ ]:
import shap
import numpy as np
import scipy.sparse

# SHAP só faz sentido se tivermos o melhor modelo carregado
if 'best_model' not in dir():
    print('⚠️  Carregue o modelo primeiro (seção 5).')
else:
    clf = best_model.named_steps['classifier']
    preprocessor = best_model.named_steps['preprocessor']

    if not hasattr(clf, 'coef_'):
        print('ℹ️  SHAP LinearExplainer requer modelo com coef_ (LogReg ou LinearSVC).')
        print(f'  Modelo atual: {type(clf).__name__} — use feature_importances_ (seção 6)')
    else:
        # Transforma o test set para matriz esparsa TF-IDF
        X_test_transformed = preprocessor.transform(X_test)

        # Garante formato esparso (necessário para LinearExplainer ser eficiente)
        if not scipy.sparse.issparse(X_test_transformed):
            X_test_transformed = scipy.sparse.csr_matrix(X_test_transformed)

        feature_names = preprocessor.get_feature_names_out()
        class_order = params['base']['class_order']

        # LinearExplainer: eficiente para modelos lineares, suporta sparse
        explainer = shap.LinearExplainer(clf, X_test_transformed, feature_perturbation='interventional')
        shap_values = explainer(X_test_transformed)

        print(f'SHAP values calculados: {shap_values.values.shape}')
        print(f'Shape: (n_samples={shap_values.values.shape[0]}, n_features={shap_values.values.shape[1]})')
        if hasattr(clf, 'classes_'):
            print(f'Classes do modelo: {list(clf.classes_)}')

## 8. Próximos passos

| Tier | Notebook | O que muda |
|------|----------|------------|
| Classic (este) | NB03 | TF-IDF + LogReg/SVM/RF — baseline interpretável |
| spaCy | NB04 | textcat treinado em português — NLP nativo |
| LLM | NB05 | Zero-shot e few-shot com Claude — sem treino |
| Benchmark | NB06 | Mesmo test set, mesmas métricas — comparação justa |

**Pergunta para o próximo notebook:** o spaCy textcat bate o recall@crítico do melhor baseline com menos dados de treino?